# Temporal Crop Experiments

Ce notebook reprend le script `temporal_crop_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Explore des modeles crop temporels compatibles avec une fenetre recente en streaming.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Train short temporal crop models for attention and blouse/PPE.
- Commande de reproduction referencee : temporal crop catalogue.
- Artefacts controles : Short temporal crop/clip catalogue exists. (`runs/exp_031_temporal_crop_catalogue/metrics/temporal_crop_metrics.csv`).
- Run par defaut : `runs/exp_031_temporal_crop_catalogue`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "temporal_crop_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import random
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from crop_cnn_experiments import TARGETS
from ml_pipeline import ROOT, safe_auc, write_json
from sequence_experiments import append_report, make_run_dir


## Fonction `set_seed`

Cette cellule definit `set_seed`. Elle prepare une partie du script.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## Fonction `resolve_run`

Cette cellule definit `resolve_run`. Elle prepare une partie du script.

In [ ]:
def resolve_run(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `build_sequence_index`

Cette cellule definit `build_sequence_index`. Elle prepare une partie du script.

In [ ]:
def build_sequence_index(crop_run, run_dir, seq_len):
    crop_run = resolve_run(crop_run)
    index = pd.read_csv(crop_run / "features" / "crop_cnn_index.csv")
    rows = []
    for video_id, group in index.groupby("video_id", sort=False):
        group = group.sort_values("time_s").reset_index(drop=True)
        if group.empty:
            continue
        for end_pos in range(len(group)):
            start_pos = max(0, end_pos - seq_len + 1)
            seq = group.iloc[start_pos : end_pos + 1]
            if len(seq) < seq_len:
                pad = pd.concat([seq.iloc[[0]]] * (seq_len - len(seq)), ignore_index=True)
                seq = pd.concat([pad, seq], ignore_index=True)
            end = group.iloc[end_pos]
            rows.append(
                {
                    "video_id": video_id,
                    "split": end["split"],
                    "end_frame": int(end["frame"]),
                    "end_time_s": float(end["time_s"]),
                    "paths": "|".join(str(p) for p in seq["path"].tolist()),
                    "frames": "|".join(str(int(f)) for f in seq["frame"].tolist()),
                    "times_s": "|".join(f"{float(t):.3f}" for t in seq["time_s"].tolist()),
                    "attention": end["attention"],
                    "attention_label": int(end["attention_label"]),
                    "blouse": end["blouse"],
                    "blouse_label": int(end["blouse_label"]),
                }
            )
    seq_index = pd.DataFrame(rows)
    seq_index.to_csv(run_dir / "features" / "temporal_crop_index.csv", index=False)
    audit = {
        "crop_run": str(crop_run),
        "seq_len": int(seq_len),
        "samples": int(len(seq_index)),
        "videos": int(seq_index["video_id"].nunique()) if len(seq_index) else 0,
        "split_counts": dict(Counter(seq_index["split"])) if len(seq_index) else {},
        "attention_counts": dict(Counter(seq_index["attention"])) if len(seq_index) else {},
        "blouse_counts": dict(Counter(seq_index["blouse"])) if len(seq_index) else {},
        "split_policy": "parent-video split inherited from crop_cnn_index.csv",
    }
    write_json(run_dir / "metrics" / "temporal_crop_dataset_audit.json", audit)
    append_report(
        run_dir,
        "Temporal Crop Dataset Build",
        "\n".join(
            [
                f"- Source crop run: `{crop_run}`",
                f"- Sequence length: `{seq_len}` sampled crop frames",
                f"- Samples: `{audit['samples']}`",
                f"- Videos: `{audit['videos']}`",
                f"- Split counts: `{audit['split_counts']}`",
                "- Split policy: parent-video split is inherited; derived crop sequences are not randomly split.",
            ]
        ),
    )
    return crop_run, seq_index


## Classe `TemporalCropDataset`

Cette cellule definit `TemporalCropDataset`. Elle prepare une partie du script.

In [ ]:
class TemporalCropDataset(Dataset):
    def __init__(self, crop_run, index, target, train=False, image_size=112, temporal_aug=True):
        self.crop_run = Path(crop_run)
        self.index = index.reset_index(drop=True)
        self.target = target
        self.train = train
        self.temporal_aug = temporal_aug and train
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        if train:
            self.tf = transforms.Compose(
                [
                    transforms.RandomResizedCrop(image_size, scale=(0.80, 1.0), ratio=(0.80, 1.25)),
                    transforms.RandomHorizontalFlip(p=0.25),
                    transforms.RandomApply([transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.18, hue=0.03)], p=0.75),
                    transforms.RandomRotation(degrees=6),
                    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.20),
                    transforms.ToTensor(),
                    normalize,
                ]
            )
        else:
            self.tf = transforms.Compose([transforms.Resize((image_size, image_size)), transforms.ToTensor(), normalize])

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        row = self.index.iloc[idx]
        tensors = []
        paths = str(row["paths"]).split("|")
        for rel_path in paths:
            image = Image.open(self.crop_run / rel_path).convert("RGB")
            tensors.append(self.tf(image))
        x = torch.stack(tensors, dim=0)
        if self.temporal_aug and len(paths) > 1:
            if random.random() < 0.25:
                drop_idx = random.randrange(x.shape[0])
                replacement = max(0, drop_idx - 1)
                x[drop_idx] = x[replacement]
            if random.random() < 0.20:
                noise = torch.randn_like(x) * 0.015
                x = x + noise
        y = torch.tensor(float(row[f"{self.target}_label"]), dtype=torch.float32)
        return x, y


## Classe `FrameEncoder`

Cette cellule definit `FrameEncoder`. Elle prepare une partie du script.

In [ ]:
class FrameEncoder(nn.Module):
    def __init__(self, embedding_dim=96):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 24, 3, padding=1),
            nn.BatchNorm2d(24),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding=1),
            nn.BatchNorm2d(48),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(48, 96, 3, padding=1),
            nn.BatchNorm2d(96),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(nn.Flatten(), nn.Dropout(0.20), nn.Linear(96, embedding_dim), nn.ReLU())

    def forward(self, frames):
        b, t, c, h, w = frames.shape
        x = frames.reshape(b * t, c, h, w)
        z = self.proj(self.features(x))
        return z.reshape(b, t, -1)


## Classe `TemporalMeanModel`

Cette cellule definit `TemporalMeanModel`. Elle prepare une partie du script.

In [ ]:
class TemporalMeanModel(nn.Module):
    def __init__(self, embedding_dim=96):
        super().__init__()
        self.encoder = FrameEncoder(embedding_dim)
        self.head = nn.Sequential(nn.Dropout(0.35), nn.Linear(embedding_dim, 1))

    def forward(self, x):
        z = self.encoder(x)
        return self.head(z.mean(dim=1)).squeeze(1)


## Classe `TemporalFlatModel`

Cette cellule definit `TemporalFlatModel`. Elle prepare une partie du script.

In [ ]:
class TemporalFlatModel(nn.Module):
    def __init__(self, seq_len, embedding_dim=96):
        super().__init__()
        self.encoder = FrameEncoder(embedding_dim)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(seq_len * embedding_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.head(self.encoder(x)).squeeze(1)


## Classe `TemporalGRUModel`

Cette cellule definit `TemporalGRUModel`. Elle prepare une partie du script.

In [ ]:
class TemporalGRUModel(nn.Module):
    def __init__(self, embedding_dim=96):
        super().__init__()
        self.encoder = FrameEncoder(embedding_dim)
        self.gru = nn.GRU(embedding_dim, 96, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Dropout(0.35), nn.Linear(192, 1))

    def forward(self, x):
        z = self.encoder(x)
        out, _ = self.gru(z)
        return self.head(out[:, -1]).squeeze(1)


## Classe `TemporalTCNModel`

Cette cellule definit `TemporalTCNModel`. Elle prepare une partie du script.

In [ ]:
class TemporalTCNModel(nn.Module):
    def __init__(self, embedding_dim=96):
        super().__init__()
        self.encoder = FrameEncoder(embedding_dim)
        self.tcn = nn.Sequential(
            nn.Conv1d(embedding_dim, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Conv1d(128, 128, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(0.35), nn.Linear(128, 1))

    def forward(self, x):
        z = self.encoder(x).transpose(1, 2)
        return self.head(self.tcn(z)).squeeze(1)


## Fonction `make_model`

Cette cellule definit `make_model`. Elle prepare une partie du script.

In [ ]:
def make_model(architecture, seq_len):
    if architecture == "mean_pool":
        return TemporalMeanModel()
    if architecture == "flat_mlp":
        return TemporalFlatModel(seq_len)
    if architecture == "gru":
        return TemporalGRUModel()
    if architecture == "tcn":
        return TemporalTCNModel()
    raise ValueError(architecture)


## Fonction `predict`

Cette cellule definit `predict`. Elle prepare une partie du script.

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    probs = []
    ys = []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb).view(-1)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        ys.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(ys)


## Fonction `best_binary_metrics`

Cette cellule definit `best_binary_metrics`. Elle prepare une partie du script.

In [ ]:
def best_binary_metrics(y, p):
    best = None
    for threshold in np.arange(0.05, 1.0, 0.05):
        pred = (p >= threshold).astype(int)
        candidate = {
            "threshold": float(threshold),
            "f1": float(f1_score(y, pred, zero_division=0)),
            "accuracy": float(accuracy_score(y, pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)) if len(np.unique(y)) > 1 else None,
            "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
        }
        if best is None or candidate["f1"] > best["f1"]:
            best = candidate
    return best


## Fonction `evaluate_predictions`

Cette cellule definit `evaluate_predictions`. Elle prepare une partie du script.

In [ ]:
def evaluate_predictions(pred_df, target):
    rows = []
    for level in ["sequence", "video"]:
        for split, group in pred_df.groupby("split"):
            if level == "sequence":
                y = group[f"{target}_label"].astype(int).to_numpy()
                p = group["risk"].to_numpy()
                n = len(group)
                positives = int(y.sum())
            else:
                video = group.groupby("video_id", as_index=False).agg(label=(f"{target}_label", "max"), risk=("risk", "mean"))
                y = video["label"].astype(int).to_numpy()
                p = video["risk"].to_numpy()
                n = len(video)
                positives = int(y.sum())
            best = best_binary_metrics(y, p)
            rows.append(
                {
                    "level": level,
                    "split": split,
                    "n": int(n),
                    "positives": positives,
                    "average_precision": safe_auc(average_precision_score, y, p),
                    "roc_auc": safe_auc(roc_auc_score, y, p),
                    **best,
                }
            )
    return rows


## Fonction `train_one`

Cette cellule definit `train_one`. Elle prepare une partie du script.

In [ ]:
def train_one(run_dir, crop_run, index, target, architecture, args, device):
    train_df = index[index["split"] == "train"].copy()
    val_df = index[index["split"] == "val"].copy()
    train_ds = TemporalCropDataset(crop_run, train_df, target, train=True, image_size=args.image_size, temporal_aug=not args.no_temporal_aug)
    val_ds = TemporalCropDataset(crop_run, val_df, target, train=False, image_size=args.image_size, temporal_aug=False)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=0)
    model = make_model(architecture, args.seq_len).to(device)
    y_train = train_df[f"{target}_label"].to_numpy()
    positives = max(1, int(y_train.sum()))
    negatives = max(1, int(len(y_train) - positives))
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([negatives / positives], dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, args.epochs))
    best = {"ap": -1.0, "state": None, "epoch": 0}
    history = []
    patience_left = args.patience
    start = time.perf_counter()
    for epoch in range(1, args.epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb).view(-1), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        scheduler.step()
        val_probs, val_y = predict(model, val_loader, device)
        val_ap = safe_auc(average_precision_score, val_y.astype(int), val_probs)
        val_score = float(val_ap or 0.0)
        history.append(
            {
                "target": target,
                "architecture": architecture,
                "epoch": epoch,
                "train_loss": float(np.mean(losses)),
                "val_ap": val_score,
                "lr": float(scheduler.get_last_lr()[0]),
            }
        )
        if val_score > best["ap"] + 1e-5:
            best = {"ap": val_score, "state": {k: v.detach().cpu() for k, v in model.state_dict().items()}, "epoch": epoch}
            patience_left = args.patience
        else:
            patience_left -= 1
        if patience_left <= 0:
            break
    if best["state"] is not None:
        model.load_state_dict(best["state"])
    train_time_s = time.perf_counter() - start
    model_path = run_dir / "models" / f"{target}_{architecture}.pt"
    torch.save(
        {
            "target": target,
            "architecture": architecture,
            "state_dict": model.state_dict(),
            "best_epoch": best["epoch"],
            "seq_len": args.seq_len,
            "image_size": args.image_size,
        },
        model_path,
    )
    return model, history, train_time_s, model_path.stat().st_size


## Fonction `make_sequence_contact_sheet`

Cette cellule definit `make_sequence_contact_sheet`. Elle prepare une partie du script.

In [ ]:
def make_sequence_contact_sheet(run_dir, crop_run, index):
    out_dir = run_dir / "error_review" / "temporal_crop_examples"
    out_dir.mkdir(parents=True, exist_ok=True)
    sample = index[index["split"] == "train"].head(8)
    rows = []
    for _, row in sample.iterrows():
        tiles = []
        for rel_path in str(row["paths"]).split("|"):
            img = cv2.imread(str(crop_run / rel_path))
            if img is None:
                continue
            img = cv2.resize(img, (112, 112))
            tiles.append(img)
        if not tiles:
            continue
        strip = np.hstack(tiles)
        cv2.putText(strip, str(row["attention"])[:18], (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(strip, str(row["blouse"])[:18], (6, 104), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 2, cv2.LINE_AA)
        rows.append(strip)
    if rows:
        max_w = max(row.shape[1] for row in rows)
        padded = []
        for row in rows:
            if row.shape[1] < max_w:
                pad = np.full((row.shape[0], max_w - row.shape[1], 3), 255, dtype=row.dtype)
                row = np.hstack([row, pad])
            padded.append(row)
        cv2.imwrite(str(out_dir / "temporal_crop_training_examples.jpg"), np.vstack(padded))


## Fonction `run_catalogue`

Cette cellule definit `run_catalogue`. Elle prepare une partie du script.

In [ ]:
def run_catalogue(args):
    set_seed(args.seed)
    run_dir = make_run_dir(args.run_name)
    write_json(
        run_dir / "config.json",
        {
            "crop_run": args.crop_run,
            "seed": args.seed,
            "seq_len": args.seq_len,
            "architectures": args.architectures,
            "epochs": args.epochs,
            "image_size": args.image_size,
            "temporal_aug": not args.no_temporal_aug,
            "created_at": datetime.now().isoformat(timespec="seconds"),
        },
    )
    crop_run, index = build_sequence_index(args.crop_run, run_dir, args.seq_len)
    make_sequence_contact_sheet(run_dir, crop_run, index)
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    all_history = []
    all_metrics = []
    all_predictions = []
    for target in TARGETS:
        for architecture in args.architectures:
            print(f"training temporal crop {target} {architecture} on {device}")
            model, history, train_time_s, model_size = train_one(run_dir, crop_run, index, target, architecture, args, device)
            all_history.extend(history)
            eval_ds = TemporalCropDataset(crop_run, index, target, train=False, image_size=args.image_size, temporal_aug=False)
            eval_loader = DataLoader(eval_ds, batch_size=args.batch_size, shuffle=False, num_workers=0)
            probs, _ = predict(model, eval_loader, device)
            pred_df = index[["video_id", "split", "end_frame", "end_time_s", f"{target}_label"]].copy()
            pred_df["target"] = target
            pred_df["architecture"] = architecture
            pred_df["risk"] = probs
            pred_df.to_csv(run_dir / "features" / f"predictions_{target}_{architecture}.csv", index=False)
            all_predictions.append(pred_df)
            for row in evaluate_predictions(pred_df, target):
                row.update(
                    {
                        "target": target,
                        "architecture": architecture,
                        "train_time_s": float(train_time_s),
                        "model_size_bytes": int(model_size),
                    }
                )
                all_metrics.append(row)
            pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "temporal_crop_training_history.csv", index=False)
            pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "temporal_crop_metrics.csv", index=False)
    if all_predictions:
        pd.concat(all_predictions, ignore_index=True).to_csv(run_dir / "features" / "temporal_crop_all_predictions.csv", index=False)
    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "temporal_crop_metrics.csv", index=False)
    lines = ["# Temporal Crop Catalogue", ""]
    lines.append("Short crop sequences test whether attention/blouse benefit from clip-level temporal context instead of independent still images.")
    lines.append("")
    lines.append("| target | architecture | level | split | AP | ROC AUC | best F1 | threshold | balanced accuracy |")
    lines.append("|---|---|---|---|---:|---:|---:|---:|---:|")
    sort_cols = ["target", "level", "split", "average_precision"]
    for _, row in metrics.sort_values(sort_cols, ascending=[True, True, True, False]).iterrows():
        lines.append(
            f"| {row['target']} | {row['architecture']} | {row['level']} | {row['split']} | "
            f"{row['average_precision'] if pd.notna(row['average_precision']) else 'NA'} | "
            f"{row['roc_auc'] if pd.notna(row['roc_auc']) else 'NA'} | {row['f1']:.3f} | "
            f"{row['threshold']:.2f} | {row['balanced_accuracy'] if pd.notna(row['balanced_accuracy']) else 'NA'} |"
        )
    summary_path = run_dir / "temporal_crop_summary.md"
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(
        run_dir,
        "Temporal Crop Catalogue Completion",
        f"- Metrics: `{run_dir / 'metrics' / 'temporal_crop_metrics.csv'}`\n- Summary: `{summary_path}`",
    )
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Train short temporal crop models for attention and blouse/PPE.")
    parser.add_argument("--crop-run", default="runs/exp_013_crop_cnn_catalogue")
    parser.add_argument("--run-name", default="exp_031_temporal_crop_catalogue")
    parser.add_argument("--architectures", nargs="+", default=["mean_pool", "flat_mlp", "gru", "tcn"])
    parser.add_argument("--seq-len", type=int, default=4)
    parser.add_argument("--image-size", type=int, default=112)
    parser.add_argument("--epochs", type=int, default=14)
    parser.add_argument("--patience", type=int, default=4)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--lr", type=float, default=4e-4)
    parser.add_argument("--weight-decay", type=float, default=2e-4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--no-temporal-aug", action="store_true")
    args = parser.parse_args()
    run_catalogue(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_031_temporal_crop_catalogue_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["temporal_crop_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
